In [25]:
import pandas as pd
import os
import json

In [26]:
# --- 1. Configuration & Setup ---
# Define directory paths
PROCESSED_DIR = '01_data/processed'
PYTHON_DIR = '04_python'
os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(PYTHON_DIR, exist_ok=True)

In [27]:
# --- 2. Data Loading & Integrity Validation ---
gateway = pd.read_csv('gateway.csv')
ledger = pd.read_csv('ledger.csv')

def validate_data(df, name):
    null_count = df.isnull().sum().sum()
    dup_count = df.duplicated().sum()
    print(f"[{name}] Nulls: {null_count}, Duplicates: {dup_count}")
    return null_count, dup_count

print("--- Data Integrity Check ---")
validate_data(gateway, "Gateway")
validate_data(ledger, "Ledger")

--- Data Integrity Check ---
[Gateway] Nulls: 0, Duplicates: 0
[Ledger] Nulls: 0, Duplicates: 0


(np.int64(0), np.int64(0))

In [28]:
# --- 3. Reconciliation Workflow ---
# Perform outer join to align all transactions by ID
recon = pd.merge(
    gateway,
    ledger,
    on='transaction_id',
    how='outer',
    suffixes=('_gateway', '_ledger')
)

# Optimized Vectorized Labeling
recon['reconciliation_status'] = 'Matched'

# Flags for categorization
is_missing_gateway = recon['amount_usd_gateway'].isnull()
is_missing_ledger = recon['amount_usd_ledger'].isnull()
both_present = (~is_missing_gateway) & (~is_missing_ledger)

# Condition masks
amt_diff = (recon['amount_usd_gateway'] != recon['amount_usd_ledger'])
sts_diff = (recon['status_gateway'] != recon['status_ledger'])

# Assign Statuses
recon.loc[is_missing_gateway, 'reconciliation_status'] = 'Missing in Gateway'
recon.loc[is_missing_ledger, 'reconciliation_status'] = 'Missing in Ledger'
recon.loc[both_present & amt_diff & sts_diff, 'reconciliation_status'] = 'Amount & Status Mismatch'
recon.loc[both_present & amt_diff & (~sts_diff), 'reconciliation_status'] = 'Amount Mismatch'
recon.loc[both_present & (~amt_diff) & sts_diff, 'reconciliation_status'] = 'Status Mismatch'

In [29]:
# --- 4. Discrepancy Identification & File Generation ---
# Extract subsets for specific issue reporting
issues = {
    "missing_in_gateway": recon[recon['reconciliation_status'] == 'Missing in Gateway'],
    "missing_in_ledger": recon[recon['reconciliation_status'] == 'Missing in Ledger'],
    "amount_mismatches": recon[recon['reconciliation_status'].str.contains('Amount')],
    "status_mismatches": recon[recon['reconciliation_status'].str.contains('Status')]
}

# Export CSV artifacts
for filename, df in issues.items():
    df.to_csv(f"{PROCESSED_DIR}/{filename}.csv", index=False)

recon.to_csv(f"{PROCESSED_DIR}/reconciliation_report.csv", index=False)

In [30]:
# --- 5. Metrics & JSON Generation ---
# Fill NaNs for numerical aggregation
recon_filled = recon.fillna(0)

# Calculate 'Amount at Risk' (Sum of max transaction value for all issues)
risk_mask = recon['reconciliation_status'] != 'Matched'
amount_at_risk = recon_filled.loc[risk_mask, ['amount_usd_gateway', 'amount_usd_ledger']].max(axis=1).sum()

summary_metrics = {
    "total_ledger_rows": int(len(ledger)),
    "total_gateway_rows": int(len(gateway)),
    "missing_in_gateway_count": int(len(issues["missing_in_gateway"])),
    "missing_in_ledger_count": int(len(issues["missing_in_ledger"])),
    "amount_mismatch_count": int(len(issues["amount_mismatches"])),
    "status_mismatch_count": int(len(issues["status_mismatches"])),
    "reconciliation_issue_count": int(risk_mask.sum()),
    "ledger_total_amount": float(ledger['amount_usd'].sum()),
    "gateway_total_amount": float(gateway['amount_usd'].sum()),
    "amount_at_risk": float(amount_at_risk)
}

# Save the final JSON metrics
with open(f"{PYTHON_DIR}/summary_metrics.json", 'w') as f:
    json.dump(summary_metrics, f, indent=4)

print("\n--- Process Complete ---")


--- Process Complete ---


JSON Normalization

In [31]:
# 1. Read the nested JSON file
with open('api_response_sample.json', 'r') as f:
    api_data = json.load(f)

In [36]:
# Ensure output directory exists
os.makedirs('01_data/processed', exist_ok=True)

In [32]:
# 2. Flatten it into tabular form
# record_path: the path to the list of records (settlements)
# meta: fields from the parent levels to include in each row
df_normalized = pd.json_normalize(
    api_data['batches'],
    record_path=['settlements'],
    meta=[
        'batch_id',
        ['merchant', 'merchant_id'],
        ['merchant', 'merchant_name'],
        ['merchant', 'region']
    ],
    sep='_'
)

In [33]:
# 3. Clean the column names (lowercase and underscore formatting)
df_normalized.columns = [c.lower().replace('.', '_') for c in df_normalized.columns]

In [34]:
# 4. Convert date/time fields
df_normalized['processed_at'] = pd.to_datetime(df_normalized['processed_at'])

In [35]:
# 5. Save the normalized output
df_normalized.to_csv('01_data/processed/api_normalized.csv', index=False)

print("Normalization complete. Output saved to: 01_data/processed/api_normalized.csv")
print(df_normalized.head())

Normalization complete. Output saved to: 01_data/processed/api_normalized.csv
  settlement_id  amount_usd   status              processed_at bank_name  \
0          S001      1520.5  settled 2026-03-07 08:10:00+00:00    Bank A   
1          S002       980.0  pending 2026-03-07 08:45:00+00:00    Bank A   
2          S003       640.0  settled 2026-03-07 09:15:00+00:00    Bank B   
3          S004      2100.0  settled 2026-03-07 08:20:00+00:00    Bank C   
4          S005       500.0   failed 2026-03-07 08:50:00+00:00    Bank C   

  bank_country batch_id merchant_merchant_id merchant_merchant_name  \
0           IN     B001                 M001             Alpha Mart   
1           IN     B001                 M001             Alpha Mart   
2           SG     B001                 M001             Alpha Mart   
3           US     B002                 M004          Delta Travels   
4           US     B002                 M004          Delta Travels   

  merchant_region  
0            APAC 

In [37]:
from google.colab import files
files.download('01_data/processed/api_normalized.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>